# Policy Comparison

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/antimicrobial-resistance=/path/to/antimicrobial-resistance"` at the top of any cell.

Compare four prescribing policies for antimicrobial stewardship:
- **Baseline**: constant cephalosporin rate (0.3)
- **Cycling**: quarterly alternation between high (0.3) and low (0.05) rates
- **Threshold**: adaptive switch to low rate when resistance exceeds 15%
- **Restriction**: linear ramp-down from 0.3 to 0.1 over 26 time steps

All simulations use the learned posterior mean parameters from Phase 3 inference.

**Prerequisites:**
- Run each policy simulation to produce `dat/policy_*_output.log`:
```bash
go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_policy_baseline.yaml
go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_policy_cycling.yaml
go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_policy_threshold.yaml
go run github.com/umbralcalc/stochadex/cmd/stochadex --config cfg/amr_policy_restriction.yaml
```

In [ ]:
import (
	"fmt"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	"github.com/umbralcalc/stochadex/pkg/simulator"
)

type policyLog struct {
	name    string
	logFile string
}

var policies = []policyLog{
	{"Baseline", "../dat/policy_baseline_output.log"},
	{"Cycling", "../dat/policy_cycling_output.log"},
	{"Threshold", "../dat/policy_threshold_output.log"},
	{"Restriction", "../dat/policy_restriction_output.log"},
}

func loadAllPolicies() map[string]*simulator.StateTimeStorage {
	storages := make(map[string]*simulator.StateTimeStorage)
	for _, p := range policies {
		s, err := analysis.NewStateTimeStorageFromJsonLogEntries(p.logFile)
		if err != nil {
			fmt.Printf("Warning: could not load %s: %v\n", p.logFile, err)
			continue
		}
		storages[p.name] = s
	}
	return storages
}

// getPartitionColumn extracts a column from a partition as []float64.
func getPartitionColumn(storage *simulator.StateTimeStorage, partition string, col int) []float64 {
	ref := analysis.DataRef{PartitionName: partition, ValueIndices: []int{col}}
	return ref.GetFromStorage(storage)[0]
}

// getTimes extracts the shared time axis from storage.
func getTimes(storage *simulator.StateTimeStorage) []float64 {
	return storage.GetTimes()
}

// xyLineData builds []opts.LineData with [x, y] pairs.
func xyLineData(xs []float64, ys []float64) []opts.LineData {
	data := make([]opts.LineData, len(xs))
	for i := range xs {
		data[i] = opts.LineData{Value: []interface{}{xs[i], ys[i]}}
	}
	return data
}

%%

storages := loadAllPolicies()
fmt.Printf("Loaded %d policy logs\n", len(storages))
for _, p := range policies {
	if _, ok := storages[p.name]; ok {
		fmt.Printf("  %s: OK\n", p.name)
	}
}

## Prescribing Rate

How each policy modulates the cephalosporin prescribing rate over time.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
	"github.com/umbralcalc/stochadex/pkg/analysis"
)

%%

storages := loadAllPolicies()

// Use NewLinePlotFromPartition for the first policy (known working pattern)
line := analysis.NewLinePlotFromPartition(
	storages["Baseline"],
	analysis.DataRef{PartitionName: "prescribing", Plotting: &analysis.DataPlotting{IsTime: true}},
	[]analysis.DataRef{
		{PartitionName: "prescribing", ValueIndices: []int{0}},
	},
	nil,
)

// Add remaining policies manually
for _, p := range policies[1:] {
	s := storages[p.name]
	times := getTimes(s)
	vals := getPartitionColumn(s, "prescribing", 0)
	line.AddSeries(p.name, xyLineData(times, vals))
}

line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Cephalosporin prescribing rate by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Prescribing rate"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
)
gonb_echarts.Display(line, "width: 1024px; height: 400px; background: white;")

## Resistance Ratio

The fraction of colonised patients carrying the resistant strain: R / (S + R). This is the primary outcome — lower is better.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

storages := loadAllPolicies()

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Resistance ratio R/(S+R) by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Resistance ratio"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for _, p := range policies {
	s := storages[p.name]
	times := getTimes(s)
	sVals := getPartitionColumn(s, "colonisation", 0)
	rVals := getPartitionColumn(s, "colonisation", 1)
	ratio := make([]float64, len(sVals))
	for i := range sVals {
		if sVals[i]+rVals[i] > 0 {
			ratio[i] = rVals[i] / (sVals[i] + rVals[i])
		}
	}
	line.AddSeries(p.name, xyLineData(times, ratio))
}
gonb_echarts.Display(line, "width: 1024px; height: 400px; background: white;")

## Colonisation Dynamics

Susceptible and resistant colonisation fractions under each policy.

In [ ]:
import (
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

storages := loadAllPolicies()

// Susceptible fraction
susLine := charts.NewLine()
susLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Susceptible colonisation fraction by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Susceptible fraction"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for _, p := range policies {
	s := storages[p.name]
	times := getTimes(s)
	vals := getPartitionColumn(s, "colonisation", 0)
	susLine.AddSeries(p.name, xyLineData(times, vals))
}
gonb_echarts.Display(susLine, "width: 1024px; height: 400px; background: white;")

// Resistant fraction
resLine := charts.NewLine()
resLine.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Resistant colonisation fraction by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Resistant fraction"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for _, p := range policies {
	s := storages[p.name]
	times := getTimes(s)
	vals := getPartitionColumn(s, "colonisation", 1)
	resLine.AddSeries(p.name, xyLineData(times, vals))
}
gonb_echarts.Display(resLine, "width: 1024px; height: 400px; background: white;")

## Cumulative Resistant BSI

Cumulative count of resistant bloodstream infections over time — the key clinical outcome for comparing stewardship policies.

In [ ]:
import (
	"fmt"

	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

storages := loadAllPolicies()

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title:  "Cumulative resistant BSI count by policy",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Cumulative resistant BSI"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Time step"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true), Bottom: "5%"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
)
for _, p := range policies {
	s := storages[p.name]
	times := getTimes(s)
	bsiR := getPartitionColumn(s, "infection", 1)
	cumBSI := make([]float64, len(bsiR))
	sum := 0.0
	for i, v := range bsiR {
		sum += v
		cumBSI[i] = sum
	}
	line.AddSeries(p.name, xyLineData(times, cumBSI))
}
gonb_echarts.Display(line, "width: 1024px; height: 400px; background: white;")

// Print final totals
fmt.Println("\n--- Final Cumulative Resistant BSI ---")
for _, p := range policies {
	s := storages[p.name]
	bsiR := getPartitionColumn(s, "infection", 1)
	total := 0.0
	for _, v := range bsiR {
		total += v
	}
	fmt.Printf("  %-15s %.0f\n", p.name+":", total)
}